## Prelude

In [ ]:
import librosa
import numpy as np
import bisect
import scipy.signal
from scipy.signal import butter, lfilter, freqz
from sklearn.decomposition import PCA
import soundfile as sf


import bokeh.io
import bokeh.plotting
import bokeh.models
import bokeh.palettes
bokeh.io.output_notebook()

from IPython.display import Audio
%matplotlib widget
import matplotlib.pyplot as plt

import mido

In [ ]:
# BALAFON_TUNING = list(reversed([1385, 1122, 985, 827, 780, 672, 570, 496, 414, 388, 338, 282, 245, 205, 193, 166, 141, 121, 102, 95, 80]))
BALAFON_TUNING = [88.6, 99.7, 104.6, 127, 141.8, 165.5, 196.5, 203.6, 247.7, 282.9, 341.4, 387.8, 415.8, 496.8, 569.4, 675, 777.3, 827.9, 985.6, 1112.9, 1383.6]
MIDI_NOTE_NUMS = [36, 38, 40, 41, 43,  48, 50, 52, 53, 55,  60, 62, 64, 65, 67,  72, 74, 76, 77, 79, 84]    # 84 isn't included in the sfz
N = len(BALAFON_TUNING)

In [ ]:
COLORS = bokeh.palettes.Turbo256

## MIDI generation

In [ ]:
def generate_midi_from_peaks(all_peaks, output_file, ticks_per_beat=480, tempo=500000, note_length_ticks=100):
    """
    Generate a MIDI file from peak data with proper note lengths.

    Parameters:
    - all_peaks: list of lists, detected peaks for each bar of the balafon.
    - output_file: str, path to save the generated MIDI file.
    - ticks_per_beat: int, resolution of the MIDI file in ticks per beat.
    - tempo: int, tempo in microseconds per beat (default is 500,000 for 120 BPM).
    - note_length_ticks: int, duration of each note in ticks.
    """
    # Flatten and sort all peaks by time
    events = []
    seconds_per_beat = tempo / 1e6
    for i, peak_list in enumerate(all_peaks):
        for peak in peak_list:
            current_tick = int((peak / seconds_per_beat) * ticks_per_beat)
            # Schedule note_on and note_off events
            events.append((current_tick, 'note_on', MIDI_NOTE_NUMS[i], 64))  # Note on
            events.append((current_tick + note_length_ticks, 'note_off', MIDI_NOTE_NUMS[i], 64))  # Note off

    # Sort all events by absolute tick time
    events.sort(key=lambda x: x[0])

    # Create the MIDI file and track
    mid = mido.MidiFile(ticks_per_beat=ticks_per_beat)
    track = mido.MidiTrack()
    mid.tracks.append(track)

    # Add tempo meta message
    track.append(mido.MetaMessage('set_tempo', tempo=tempo))

    # Add events to the MIDI track with correct relative timing
    last_tick = 0
    for tick, event_type, note, velocity in events:
        delta_ticks = tick - last_tick
        last_tick = tick
        track.append(mido.Message(event_type, note=note, velocity=velocity, time=delta_ticks))

    # Save the MIDI file
    mid.save(output_file)

## Process/Clean sensor data files

### Process data files

In [ ]:
def twos_complement(hexstr, bits):
    value = int(hexstr, 16)
    if value & (1 << (bits - 1)):
        value -= 1 << bits
    return value

def read_sensor_data_hex(filename, v=False):
  # times, x, y, z
  data = [([], [], [], []) for _ in range(N)]
  interrupt_timestamps = [[] for _ in range(N)]

  with open(filename, "r") as f:
    for row in f:
      values = row[:-1].split(",")
      if(len(values) != 6):
        if v:
          print("Wrong number of values")
          print(row)
        continue

      timestamp = int(values[0], 16)
      sensor_number = int(values[1], 16)
      x_accel = twos_complement(values[2], 32)
      y_accel = twos_complement(values[3], 32)
      z_accel = twos_complement(values[4], 32)
      interrupt = int(values[5], 16)

      for val in [x_accel, y_accel, z_accel]:
        if (abs(val) > 40000):
          if v:
            print("Out of range")
            print(row[:-1])
          continue

      last_timestamp = None
      if (last_timestamp and timestamp <= last_timestamp):
        if v:
          print("Out of order")
          print(row[:-1])
        continue
      else:
        last_timestamp = timestamp

      if (sensor_number >= N and sensor_number < 0):
        if v:
          print(row[:-1])
        continue

      sensor_data = data[sensor_number]
      sensor_data[0].append(timestamp)
      sensor_data[1].append(x_accel)
      sensor_data[2].append(y_accel)
      sensor_data[3].append(z_accel)

      if (interrupt != 0):
        interrupt_timestamps[sensor_number].append(interrupt)

  return data, interrupt_timestamps

def detect_outliers(data, method="zscore", threshold=3, v=False):
    """
    Detect outliers in the data using Z-score or IQR and print information about them.

    Parameters:
    - data: numpy array, the input data.
    - method: str, the method to use for outlier detection ("zscore" or "iqr").
    - threshold: float, the threshold for detecting outliers.

    Returns:
    - mask: numpy array, a boolean mask where True indicates non-outliers.
    """
    data = np.array(data)
    if method == "zscore":
        # Z-score method
        mean = np.mean(data)
        std = np.std(data)
        z_scores = (data - mean) / std
        mask = np.abs(z_scores) < threshold
        outliers = np.where(~mask)[0]
        if v:
          print(f"Z-score Outliers Detected: {len(outliers)}")
          for idx in outliers:
              print(f"Index: {idx}, Value: {data[idx]}, Z-score: {z_scores[idx]}")
    elif method == "iqr":
        # IQR method
        q1 = np.percentile(data, 25)
        q3 = np.percentile(data, 75)
        iqr = q3 - q1
        lower_bound = q1 - threshold * iqr
        upper_bound = q3 + threshold * iqr
        mask = (data >= lower_bound) & (data <= upper_bound)
        outliers = np.where(~mask)[0]
        if v: 
          print(f"IQR Outliers Detected: {len(outliers)}")
          for idx in outliers:
              print(f"Index: {idx}, Value: {data[idx]}, Bounds: ({lower_bound}, {upper_bound})")
    else:
        raise ValueError("Invalid method. Use 'zscore' or 'iqr'.")
    
    return mask

def extract_and_clean_sensor_data(datafile, outlier_method="zscore", outlier_threshold=100, v=False):
    all_data, interrupt_timestamps = read_sensor_data_hex(datafile, v=v)

    # Apply outlier detection and cleaning
    cleaned_data = []
    for sensor_data in all_data:
        mask = detect_outliers(np.linalg.norm(sensor_data[1:], axis=0), method=outlier_method, threshold=outlier_threshold, v=v)
        timestamps, xx, yy, zz = sensor_data
        timestamps = np.array(timestamps)
        xx = np.array(xx)
        yy = np.array(yy)
        zz = np.array(zz)
        # Apply mask to timestamps and accelerometer data
        cleaned_data.append([timestamps[mask], xx[mask], yy[mask], zz[mask]])

    return cleaned_data


### Process timestamps

In [ ]:
def normalize_timestamps(data):
    """
    Normalize timestamps in the data to start from 0.

    Parameters:
    - data: list of lists, where each inner list contains timestamps and accelerometer data.

    Returns:
    - data: list of lists with normalized timestamps.
    """
    min_timestamp = min([min(sensor_data[0]) for sensor_data in data])
    for sensor_data in data:
        timestamps = sensor_data[0]
        if len(timestamps) == 0:
            continue
        sensor_data[0] = [(ts - min_timestamp) / 1e6 for ts in timestamps]
    
    return data

### Test

In [ ]:
# Clean sensor data and plot first dimension of first sensor
data = normalize_timestamps(extract_and_clean_sensor_data("./data/test2", v=True, outlier_threshold=35))
plt.plot(data[1][0], data[1][1], label='X-axis')

In [ ]:
max_timestamp = max([max(sensor_data[0]) for sensor_data in data])
max_timestamp

## Process (aligned) audio

In [ ]:
def cqt(audio, center_freqs, sr=None, hop_length=32, bins_per_octave=20):
    """
    Compute the Constant-Q Transform (CQT) of an audio signal and return the CQT matrix.

    Parameters:
    - audio: audio signal as a numpy array
    - center_freqs: list of floats, specific center frequencies for the filterbank.
    - sr: sampling rate of the audio
    - hop_length: int, number of samples between successive frames for the CQT.
    - bins_per_octave: int, number of bins per octave in the CQT.

    Returns:
    - timestamps: numpy array, timestamps corresponding to the CQT channels.
    - cqt_channels: list of numpy arrays, CQT channels corresponding to the specified center frequencies.
    """
    n_bins = bins_per_octave * int(np.ceil(np.log2(max(center_freqs) / min(center_freqs))))
    cqt = librosa.cqt(
        audio, sr=sr, 
        hop_length=hop_length, 
        fmin=min(center_freqs), 
        n_bins=n_bins, 
        bins_per_octave=bins_per_octave)
    cqt_frequencies = librosa.cqt_frequencies(cqt.shape[0], fmin=min(center_freqs), bins_per_octave=bins_per_octave)
    
    cqt_channels = []
    for freq in center_freqs:
        if freq not in cqt_frequencies:
            closest_bin = np.argmin(np.abs(cqt_frequencies - freq))
            cqt_channels.append(cqt[closest_bin])
    timestamps = np.arange(len(cqt_channels[0])) * hop_length / sr
    return timestamps, cqt_channels


### Test

In [ ]:
plt.cla()
audio, sr = librosa.load("./audio/ZOOM0004_INPUT1_TRIMMED.WAV")
timestamps, cqt_channels = cqt(audio, BALAFON_TUNING, sr=sr)
plt.plot(timestamps, cqt_channels[11], label='CQT Channel 11')
plt.show()

## Signal transformations

In [ ]:
def window_rms(a, window_size):
  a2 = np.power(a,2)
  window = np.ones(window_size)/float(window_size)
  return np.sqrt(np.convolve(a2, window, 'valid'))   # same keeps border effects

# https://kferg.dev/posts/2020/audio-reactive-programming-envelope-followers/
def envelope_follower(signal, sample_rate, attack_time_ms, release_time_ms):
    # Convert attack/release times to filter coefficients
    alpha_attack = np.exp(-1 / (sample_rate * attack_time_ms / 1000))
    alpha_release = np.exp(-1 / (sample_rate * release_time_ms / 1000))

    envelope = np.zeros_like(signal, dtype=np.double)
    current_envelope = 0.0

    for i in range(len(signal)):
        abs_sample = abs(signal[i])
        if abs_sample > current_envelope:
            current_envelope = alpha_attack * current_envelope + (1 - alpha_attack) * abs_sample
        else:
            current_envelope = alpha_release * current_envelope + (1 - alpha_release) * abs_sample
        envelope[i] = current_envelope
    return envelope

def sensor_data_transformations(data, methods=[], outlier_method="zscore", 
                                  outlier_threshold=100, attack_time_ms=5, release_time_ms=500, v=False):
  transformed_signals = []
  mag = np.sqrt(np.array(data[1])**2 + np.array(data[2])**2 + np.array(data[3])**2)
  mask = detect_outliers(mag, method=outlier_method, threshold=outlier_threshold, v=v)
  mag_cleaned = mag[mask]
  timestamps_cleaned = np.array(data[0])[mask]
  x_cleaned = np.array(data[1], dtype=np.double)[mask]
  y_cleaned = np.array(data[2], dtype=np.double)[mask]
  z_cleaned = np.array(data[3], dtype=np.double)[mask]

  # Remove DC offset
  x_cleaned -= np.median(x_cleaned)
  y_cleaned -= np.median(y_cleaned)
  z_cleaned -= np.median(z_cleaned)

  for method in methods:
    if method == "raw_x":
      transformed_signals.append((timestamps_cleaned, x_cleaned))
    elif method == "raw_y":
      transformed_signals.append((timestamps_cleaned, y_cleaned))
    elif method == "raw_z":
      transformed_signals.append((timestamps_cleaned, z_cleaned))
    elif method == "magnitude":
      transformed_signals.append((timestamps_cleaned, mag_cleaned))
    elif method == "rms":
      rms = window_rms(mag_cleaned, 64)
      rms -= np.median(rms)
      rms /= np.max(rms)
      timestamps_cleaned = timestamps_cleaned[:len(rms)]  # Adjust timestamps to match RMS length
      transformed_signals.append((timestamps_cleaned, rms))
    elif method == "pca":
      pca = PCA(n_components=1, svd_solver='full')
      # Stack the x, y, z data for PCA
      data_matrix = np.vstack((x_cleaned, y_cleaned, z_cleaned)).T
      pca_result = pca.fit_transform(data_matrix)
      if v:
        print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
      transformed_signals.append((timestamps_cleaned, pca_result.flatten()))
  return transformed_signals

def normalize_signal(signal, robust=True, dc_offset=True, percentile=99):
    signal = signal.copy()
    if dc_offset:
        signal -= np.median(signal)
    if robust:
        max_val = np.percentile(np.abs(signal), percentile)
    else:
        max_val = np.max(np.abs(signal))
    signal /= max_val
    return signal

## Audio peak finding

In [ ]:
AUDIO_ENV_PEAK_PARAMS = dict(pre_max=500, post_max=500, pre_avg=500, post_avg=500, delta=0.001, wait=100, threshold=0.1)

In [ ]:
def find_peaks(signal, peak_params={}):
    peak_params = peak_params.copy()
    threshold = peak_params.pop('threshold')
    peaks = librosa.util.peak_pick(signal, **peak_params)

    valid_peaks = []
    for peak in peaks:
        if signal[peak] < threshold:
            continue
        valid_peaks.append(peak)
    return valid_peaks


### Test

In [ ]:
audio_envelope = envelope_follower(audio, sr, attack_time_ms=1, release_time_ms=20)
normalized_audio_envelope = normalize_signal(audio_envelope, dc_offset=False)
x_envelope = np.linspace(0, len(audio) / sr, len(audio_envelope))
x_audio = np.linspace(0, len(audio) / sr, len(audio))

peaks = find_peaks(normalized_audio_envelope, peak_params=AUDIO_ENV_PEAK_PARAMS)


In [ ]:
plt.cla()
plt.plot(x_envelope, normalized_audio_envelope, label='Normalized Audio Envelope')
plt.plot(x_audio, audio, label='Audio Signal', alpha=0.5)

# draw rectangles around the peaks showing the pre and post max
for peak in peaks:
    plt.axvspan(x_envelope[peak] - AUDIO_ENV_PEAK_PARAMS['pre_max'] / sr,
                x_envelope[peak] + AUDIO_ENV_PEAK_PARAMS['post_max'] / sr,
                color='red', alpha=0.1)

plt.vlines(x_envelope[peaks], ymin=0, ymax=1, color='red', label='Peaks', alpha=0.5)

plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

## Sensor peakfinding

In [ ]:
# TODO: set relative to sampling rate
SENSOR_ENV_PEAK_PARAMS = {
    'pre_max': 30,
    'post_max': 1,
    'pre_avg': 30,
    'post_avg': 1,
    'delta': 0.1,
    'wait': 20,
    'threshold': 0
}

SENSOR_PROCESSING_PARAMS = {
    'method': 'pca',
    'norm_percentile': 99.99,
    'env_attack_time_ms': 1,
    'env_release_time_ms': 150,
}

REGION_FINDING_PARAMS = {
    'threshold': 0.4,
    'pre_padding': 5,
    'post_padding': 15
}

ACTIVE_REGION_THRESHOLD = 0.8

In [ ]:
class SensorEvent:
    def __init__(self, i, index, time, amplitude):
        self.i = i
        self._index = index
        self.time = time
        self.amplitude = amplitude

class SensorRegion:
    def __init__(self, i, start, end):
        self.i = i
        self.start = start
        self.end = end
        self.events = []

    def add_events(self, events):
        self.events.extend(events)
        self.events.sort(key=lambda e: e._index)

    def primary_event(self):
        if not self.events:
            return None
        return self.events[0]   # TODO: or max amplitude?
    
    def duration(self):
        return self.end - self.start

class SensorProcess:
    def __init__(self, sensor_num, data):
        self.i = sensor_num
        self.raw_data = data

    def process(self, params=SENSOR_PROCESSING_PARAMS):
        sensor_transformations = sensor_data_transformations(self.raw_data, methods=[params['method']])
        transformation = np.array(sensor_transformations[0])
        normalized = normalize_signal(transformation[1], robust=True, dc_offset=False, percentile=params['norm_percentile'])
        estimated_sr = 1 / np.mean(np.diff(transformation[0])) # 364.63 Hz, 0.01 Hz standard deviation
        sensor_envelope = envelope_follower(normalized, estimated_sr, 
                                       attack_time_ms=params['env_attack_time_ms'], 
                                       release_time_ms=params['env_release_time_ms'])
        self.x = transformation[0]
        self.y = normalized
        self.estimated_sr = estimated_sr
        self.env = sensor_envelope
    
    def find_peaks(self, params=SENSOR_ENV_PEAK_PARAMS):
        self.peaks = find_peaks(self.env, peak_params=params)
        self.events = [SensorEvent(self.i, peak, self.x[peak], self.env[peak]) for peak in self.peaks]

    def find_active_regions(self, params=REGION_FINDING_PARAMS):
        """Find active regions where envelope is above a fixed threshold."""
        if not self.events:
            self.active_regions = []
            return
            
        self.active_regions = []
        
        # Find all regions above threshold
        above_threshold = self.env > params['threshold']

        if not np.any(above_threshold):
            return
        
        # Find initial boundaries
        diff = np.diff(np.concatenate(([False], above_threshold, [False])).astype(int))
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]
        
        # Extend boundaries
        extended_starts = np.maximum(0, starts - params['pre_padding'])
        extended_ends = np.minimum(len(self.env), ends + params['post_padding'])

        # Create regions and assign events to them
        for start, end in zip(extended_starts, extended_ends):
            region = SensorRegion(self.i, start, end)
            
            # Find all events within this region
            region_events = [event for event in self.events 
                            if start <= event._index < end]
            
            if region_events:  # Only create region if it has events
                region.add_events(region_events)
                self.active_regions.append(region)


### Test

In [ ]:
plt.cla()

sensors_to_view = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
sensors_to_view = [12]
processed_sensor_data = []

for i in sensors_to_view:
    sensor = data[i - 1]  # Adjust for zero-based index
    sensor_process = SensorProcess(i, sensor)
    sensor_process.process()
    sensor_process.find_peaks()
    sensor_process.find_active_regions()

    # # plot normalized sensor data
    # plt.plot(sensor_process.x, sensor_process.y, label=f'Sensor {i}', color=COLORS[i * 10])

    # # plot sensor envelopes
    plt.plot(sensor_process.x, sensor_process.env, label=f'Sensor {i} Envelope', color=COLORS[i * 10 + 5])

    # plot sensor peaks
    plt.vlines(sensor_process.x[sensor_process.peaks], ymin=0, ymax=1, color=COLORS[i * 10 + 3], label=f'Sensor {i} Peaks', alpha=0.5, linewidth=1)

    # plot active regions with a diagonal line through it
    for region in sensor_process.active_regions:
        plt.axvspan(sensor_process.x[region.start], sensor_process.x[region.end], color=COLORS[i * 10 + 1], alpha=0.3)
        # diagonal lines through region
        plt.plot([sensor_process.x[region.start], sensor_process.x[region.end]], [0, 1], color=COLORS[i * 10 + 1], alpha=0.5)

# plot audio envelope
# plt.plot(x_envelope, normalized_audio_envelope,  label='Normalized Audio Envelope', alpha=0.5)

# # plot audio
plt.plot(x_audio, audio * 0.4)

# plot audio peaks
# plt.vlines(x_envelope[peaks], ymin=0, ymax=1, color='red', label='Peaks', alpha=0.5, linewidth=0.5)


plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Cluster sensor data onsets

In [ ]:
CLUSTER_MAX_GAP = 0.05

In [ ]:
class SensorEventCluster:
    def __init__(self, event):
        self.events = [event]

    def add_event(self, event):
        self.events.append(event)
    
    def last_time(self):
        return self.events[-1].time
    
    def first_time(self):
        return self.events[0].time

    def highest_amplitude_event(self):
        return max(self.events, key=lambda x: x.amplitude)

    def copy(self):
        new_cluster = SensorEventCluster(self.events[0])
        new_cluster.events = self.events[:]
        return new_cluster

def cluster_sensor_events(sensor_events, max_gap=CLUSTER_MAX_GAP):
    if not sensor_events:
        return []

    # Sort events by start time
    sensor_events = sorted(sensor_events, key=lambda x: x.time)

    clusters = []
    current_cluster = SensorEventCluster(sensor_events[0])

    for event in sensor_events[1:]:
        # If the gap to the next event is within the max_gap, add it to the current cluster
        if event.time - current_cluster.last_time() <= max_gap:
            current_cluster.add_event(event)
        else:
            # Otherwise, finalize the current cluster and start a new one
            clusters.append(current_cluster)
            current_cluster = SensorEventCluster(event)

    clusters.append(current_cluster)

    return clusters



### Test

In [ ]:
sensor_processes = []
for i in range(N):
    sensor = data[i]
    sensor_process = SensorProcess(i, sensor)
    sensor_process.process()
    sensor_process.find_peaks()
    sensor_process.find_active_regions()
    sensor_processes.append(sensor_process)

sensor_events = []
for sp in sensor_processes:
    sensor_events.extend(sp.events)
clusters = cluster_sensor_events(sensor_events)

In [ ]:
# visualize the distribution of the amount of events per cluster and the distribution of amplitudes within each cluster
plt.cla()
# Distribution of the amount of events per cluster
cluster_sizes = [len(c.events) for c in clusters]
plt.figure(figsize=(12, 6))
plt.hist(cluster_sizes, bins=range(1, max(cluster_sizes) + 1), align='left', color='blue', alpha=0.7)
plt.title('Distribution of Events per Cluster')
plt.xlabel('Number of Events')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# plot a histogram of the indexes of the sensor with the highest amplitude

winning_sensor_indices = [cluster.highest_amplitude_event().i for cluster in clusters if cluster.highest_amplitude_event()]
plt.figure(figsize=(8, 4))
plt.hist(winning_sensor_indices, bins=np.arange(1, 23) - 0.5, edgecolor='black')
plt.xticks(np.arange(1, 22))
plt.title('Highest amplitude sensors per cluster')
plt.xlabel('Sensor Index')
plt.ylabel('Frequency')
plt.show()


In [ ]:
plt.cla()
plt.figure(figsize=(8, 4))
for i, cluster in enumerate(clusters):
    amplitudes = [event.amplitude for event in cluster.events]
    plt.plot(sorted(amplitudes), label=f'Cluster {i + 1}')
plt.title('Amplitudes within Each Cluster')
plt.xlabel('Event Index')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

In [ ]:
plt.cla()

sensors_to_view = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]


for i in sensors_to_view:
    sensor_process = sensor_processes[i - 1]  # Use the pre-processed sensor data

    # # plot normalized sensor data
    # plt.plot(sensor_process.x, sensor_process.y, label=f'Sensor {i}', color=COLORS[i * 10])

    # plot sensor envelopes
    plt.plot(sensor_process.x, sensor_process.env, label=f'Sensor {i} Envelope', color=COLORS[i * 10 + 5])

    # plot sensor peaks
    plt.vlines(sensor_process.x[sensor_process.peaks], ymin=0, ymax=1, color=COLORS[i * 10 + 3], label=f'Sensor {i} Peaks', alpha=0.5, linewidth=1)

# plot clusters
for cluster in clusters:
   plt.axvspan(cluster.first_time(), cluster.last_time(), ymin=0, ymax=1,color='gray', alpha=0.5)

# plot audio envelope
# plt.plot(x_envelope, normalized_audio_envelope,  label='Normalized Audio Envelope', alpha=0.5)

# # plot audio
# plt.plot(x_audio, audio * 0.4)

# plot audio peaks
plt.vlines(x_envelope[peaks], ymin=0, ymax=1, color='red', label='Peaks', alpha=0.5, linewidth=0.5)


plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Match audio and sensor peaks

In [ ]:
PEAK_MATCH_MAX_GAP = 0.05

In [ ]:
def find_primary_clusters(clusters, all_regions):
    # Check highest amplitude events in each clusters to see if 
    # they are primary to a sensor region
    primary_clusters = []
    other_clusters = []
    
    for cluster in clusters:
        highest_amp_event = cluster.highest_amplitude_event()
        sensor_regions = all_regions[highest_amp_event.i]
        for region in sensor_regions:
            if highest_amp_event is region.primary_event():
                primary_clusters.append(cluster)
                break
        else:
            other_clusters.append(cluster)

    return primary_clusters, other_clusters

In [ ]:
def match_audio_and_sensor_peaks(audio_peak_indexes, audio_sr, sensor_event_clusters, max_gap=PEAK_MATCH_MAX_GAP):
    matched_audio_sensor_pairs = []
    audio_peaks = list(audio_peak_indexes)  # Start with all audio peaks as unmatched
    unmatched_sensor_clusters = []

    for cluster in sensor_event_clusters:
        cluster_time = cluster.highest_amplitude_event().time
        possible_matches = []
        
        # Find all audio peaks within the time gap
        for audio_peak in audio_peaks:
            audio_time = audio_peak / audio_sr
            if abs(cluster_time - audio_time) <= max_gap:
                possible_matches.append(audio_peak)
        
        if possible_matches:
            # Find the closest audio peak
            best_match = min(possible_matches, key=lambda x: abs(cluster_time - x / audio_sr))
            matched_audio_sensor_pairs.append((best_match, cluster))
            audio_peaks.remove(best_match)  # Remove matched audio peak
        else:
            unmatched_sensor_clusters.append(cluster)

    # return matching pairs and unmatched audio peaks and sensor clusters
    return matched_audio_sensor_pairs, audio_peaks, unmatched_sensor_clusters 

### Test

In [ ]:
print("Identifying primary clusters...")
primary_clusters, other_clusters = find_primary_clusters(clusters, [sp.active_regions for sp in sensor_processes])
primary_clusters.sort(key=lambda x: len(x.events), reverse=True)    # prioritize larger clusters

total_events = sum(len(sp.events) for sp in sensor_processes)
total_regions = sum(len(sp.active_regions) for sp in sensor_processes)
print(total_events, "total events found")
print(total_regions, "total regions found")
print(len(primary_clusters), "primary clusters found")
print(len(other_clusters), "other clusters found")
print()

print("Matching audio peaks to primary clusters...")
primary_matched_peaks, unmatched_audio_peaks, unmatched_primary_sensor_clusters = match_audio_and_sensor_peaks(peaks, sr, primary_clusters)
print(len(primary_matched_peaks), "primary pairs found")
print(len(unmatched_audio_peaks), "unmatched audio peaks remaining")
print(len(unmatched_primary_sensor_clusters), "unmatched primary sensor clusters remaining")
print()

singleton_clusters = [cluster for cluster in other_clusters if len(cluster.events) == 1]
multi_event_clusters = [cluster for cluster in other_clusters if len(cluster.events) > 1]
multi_event_clusters.sort(key=lambda x: len(x.events), reverse=True)    # prioritize larger clusters
print(len(singleton_clusters), "singleton clusters among other clusters")
print(len(multi_event_clusters), "multi-event clusters among other clusters")
print()

print("Matching multi-event clusters with remaining audio peaks...")
# matching multi-event non_primary clusters with remaining audio peaks
secondary_matched_peaks, unmatched_audio_peaks, unmatched_other_sensor_clusters = match_audio_and_sensor_peaks(unmatched_audio_peaks, sr, multi_event_clusters)
print(len(secondary_matched_peaks), "matched peaks")
print(len(unmatched_audio_peaks), "unmatched audio peaks")
print(len(unmatched_other_sensor_clusters), "unmatched sensor clusters")
print()

print("Matching singleton clusters with remaining audio peaks (max_gap=0.03)...")
# matching singleton clusters with remaining audio peaks
singleton_matched_peaks, unmatched_audio_peaks, unmatched_singleton_sensor_clusters = match_audio_and_sensor_peaks(unmatched_audio_peaks, sr, singleton_clusters, max_gap=0.03)
print(len(singleton_matched_peaks), "matched peaks")
print(len(unmatched_audio_peaks), "unmatched audio peaks")
print(len(unmatched_singleton_sensor_clusters), "unmatched sensor clusters")


In [ ]:
plt.cla()
fig, ax = plt.subplots(figsize=(8, 6))

# Get cluster sizes for both groups
primary_cluster_sizes = [len(cluster.events) for cluster in primary_clusters]
other_cluster_sizes = [len(cluster.events) for cluster in other_clusters]

# Create bins that cover the range of both datasets
max_size = max(max(primary_cluster_sizes) if primary_cluster_sizes else [0], 
               max(other_cluster_sizes) if other_cluster_sizes else [0])
bins = np.arange(1, max_size + 2) - 0.4

# Create histograms with offset bars
width = 0.4
x = np.arange(1, max_size + 1)

# Count frequencies for each bin
primary_counts = np.histogram(primary_cluster_sizes, bins=np.arange(1, max_size + 2))[0]
other_counts = np.histogram(other_cluster_sizes, bins=np.arange(1, max_size + 2))[0]

# Create bars
ax.bar(x - width/2, primary_counts, width, label='Primary Clusters', alpha=0.7, color='blue')
ax.bar(x + width/2, other_counts, width, label='Other Clusters', alpha=0.7, color='red')

ax.set_xlabel('Number of Events per Cluster')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Events per Cluster: Primary vs Other Clusters')
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.cla()
plt.figure(figsize=(11, 6))

sensors_to_view = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]

for i in sensors_to_view:
    sensor_process = sensor_processes[i - 1]  # Use the pre-processed sensor data

    # # plot normalized sensor data
    # plt.plot(sensor_process.x, sensor_process.y, label=f'Sensor {i}', color=COLORS[i * 10])

    # plot sensor envelopes
    plt.plot(sensor_process.x, sensor_process.env, color=COLORS[i * 10 + 5])

    # plot sensor peaks
    # plt.vlines(sensor_process.x[sensor_process.peaks], ymin=0, ymax=1, color=COLORS[i * 10 + 3], label=f'Sensor {i} Peaks', alpha=0.5, linewidth=1)


# plot primary sensor/audio matches
for audio_peak, sensor_cluster in primary_matched_peaks:
    plt.plot([audio_peak / sr] * 2, [0, 1], linestyle=':', color='purple', alpha=0.6)    # matched audio peaked
    winning_event = sensor_cluster.highest_amplitude_event()
    plt.plot([winning_event.time] * 2, [0, 1], linestyle=':', color=COLORS[winning_event.i * 10 + 5], alpha=0.5) # matched sensor peak
    plt.axvspan(audio_peak / sr, winning_event.time, ymin=0, ymax=1, color='purple', alpha=0.5) # rectangle

# plot secondary sensor/audio matches
for audio_peak, sensor_cluster in secondary_matched_peaks:
    plt.plot([audio_peak / sr] * 2, [0, 1], linestyle=':', color='blue', alpha=0.6)    # matched audio peaked
    winning_event = sensor_cluster.highest_amplitude_event()
    plt.plot([winning_event.time] * 2, [0, 1], linestyle=':', color=COLORS[winning_event.i * 10 + 5], alpha=0.5) # matched sensor peak
    plt.axvspan(audio_peak / sr, winning_event.time, ymin=0, ymax=1, color='blue', alpha=0.5) # rectangle

for audio_peak, sensor_cluster in singleton_matched_peaks:
    plt.plot([audio_peak / sr] * 2, [0, 1], linestyle=':', color='pink', alpha=0.6)    # matched audio peaked
    winning_event = sensor_cluster.highest_amplitude_event()
    plt.plot([winning_event.time] * 2, [0, 1], linestyle=':', color=COLORS[winning_event.i * 10 + 5], alpha=0.5) # matched sensor peak
    plt.axvspan(audio_peak / sr, winning_event.time, ymin=0, ymax=1, color='pink', alpha=0.5) # rectangle

# plot audio envelope
# plt.plot(x_envelope, normalized_audio_envelope,  label='Normalized Audio Envelope', alpha=0.5)

# plot audio
plt.plot(x_audio, audio * 0.4)

# plot unmatched audio peaks
plt.vlines(np.array(unmatched_audio_peaks) / sr, ymin=0, ymax=1, color='red', label='Unmatched Audio Peaks', alpha=0.5, linewidth=0.5)

# unmatched primary sensor peaks
plt.vlines([cluster.highest_amplitude_event().time for cluster in unmatched_primary_sensor_clusters], ymin=0, ymax=1, color='black', label='Unmatched Sensor Peaks', alpha=0.5, linewidth=0.5)

# unmatched secondary sensor peaks (multi-cluster)
plt.vlines([cluster.highest_amplitude_event().time for cluster in unmatched_other_sensor_clusters], ymin=0, ymax=1, color='orange', label='Unmatched Secondary Sensor Peaks', alpha=0.5, linewidth=0.5)

# singleton clusters
plt.vlines([cluster.highest_amplitude_event().time for cluster in singleton_clusters], ymin=0, ymax=1, color='pink', label='Singleton Clusters', alpha=0.5, linewidth=0.5)


plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Generate test MIDI

In [ ]:
all_matched_peaks = primary_matched_peaks + secondary_matched_peaks + singleton_matched_peaks

sorted_matches = [[] for _ in range(N)]
for audio_peak, sensor_cluster in all_matched_peaks:
    sensor_id = sensor_cluster.highest_amplitude_event().i
    audio_time = audio_peak / sr
    sorted_matches[sensor_id].append(audio_time)

generate_midi_from_peaks(sorted_matches, "out/test4.mid")


## Investigate audio spectogram

In [ ]:
from bokeh.models import HoverTool, ColorBar
from bokeh.transform import linear_cmap
from bokeh.palettes import Viridis256
import numpy as np

# Create an interactive spectrogram using Bokeh

# Compute the spectrogram
hop_length = 512
n_fft = 2048
stft = librosa.stft(audio, hop_length=hop_length, n_fft=n_fft)
magnitude_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

# Create time and frequency axes
times = librosa.frames_to_time(np.arange(magnitude_db.shape[1]), sr=sr, hop_length=hop_length)
freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

# Create mesh grid for bokeh image
dw = times[1] - times[0]  # time step
dh = freqs[1] - freqs[0]  # frequency step

# Create the bokeh figure
p = bokeh.plotting.figure(
    title="Interactive Audio Spectrogram",
    x_axis_label="Time (s)",
    y_axis_label="Frequency (Hz)",
    width=1000,
    height=600,
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

# Add hover tool
hover = HoverTool(tooltips=[
    ("Time", "$x{0.000} s"),
    ("Frequency", "$y{0.0} Hz"),
    ("Magnitude", "@image dB")
])
p.add_tools(hover)

# Create color mapper
color_mapper = linear_cmap(field_name='image', palette=Viridis256, low=magnitude_db.min(), high=magnitude_db.max())

# Add the spectrogram as an image
p.image(image=[magnitude_db], x=times[0], y=freqs[0], dw=times[-1], dh=freqs[-1], color_mapper=color_mapper['transform'])

# Add colorbar
color_bar = ColorBar(color_mapper=color_mapper['transform'], width=8, location=(0,0))
p.add_layout(color_bar, 'right')

# Limit frequency range to something more reasonable (0-8kHz)
p.y_range.start = 0
p.y_range.end = 8000

bokeh.io.show(p)

## New section

In [ ]:
plt.plot()

In [ ]:
x = np.linspace(0, len(audio_envelope) / sr, len(audio_envelope))
y = np.sin(x)

In [ ]:
plt.cla()

In [ ]:
plt.plot(x, audio_envelope)
plt.show()

In [ ]:
# plot the audio in an interactive bokeh plot
p = bokeh.plotting.figure(title="Audio Envelope", x_axis_label='Time (s)', y_axis_label='Amplitude', width=800, height=400)
r = p.line(x, y, line_width=2, color='blue', legend_label='Audio Envelope')
r.visible = True

bokeh.io.show(p)

In [ ]:
# ...existing code...
# Debug the data first
print(f"x shape: {x.shape}, range: [{x.min():.3f}, {x.max():.3f}]")
print(f"y shape: {y.shape}, range: [{y.min():.3f}, {y.max():.3f}]")
print(f"audio_envelope shape: {audio_envelope.shape}, range: [{audio_envelope.min():.3f}, {audio_envelope.max():.3f}]")

# Plot with matplotlib first to verify data
plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.plot(x, y, 'b-', linewidth=1)
plt.title('Sine wave')
plt.subplot(2, 1, 2)
plt.plot(x, audio_envelope, 'r-', linewidth=1)
plt.title('Audio envelope')
plt.tight_layout()
plt.show()

# Then try bokeh with explicit ranges
p = bokeh.plotting.figure(
    title="Audio Envelope", 
    x_axis_label='Time (s)', 
    y_axis_label='Amplitude', 
    width=800, 
    height=400,
    x_range=(x.min(), x.max()),
    y_range=(min(y.min(), audio_envelope.min()), max(y.max(), audio_envelope.max()))
)
r = p.line(x, y, line_width=2, color='blue', legend_label='Sine Wave')
r2 = p.line(x, audio_envelope, line_width=2, color='red', legend_label='Audio Envelope')

bokeh.io.show(p)
# ...existing code...

In [ ]:
plt.plot(np.arange(len(audio_envelope)) / sr, audio_envelope, label='Audio Envelope')

In [ ]:
# ...existing code...
# Downsample data for Bokeh (take every 1000th point to reduce from 6.5M to ~6.5K points)
downsample_factor = 1000
x_ds = x[::downsample_factor]
y_ds = y[::downsample_factor]
audio_envelope_ds = audio_envelope[::downsample_factor]

print(f"Downsampled x shape: {x_ds.shape}")
print(f"Downsampled y shape: {y_ds.shape}")
print(f"Downsampled audio_envelope shape: {audio_envelope_ds.shape}")

# Try bokeh with downsampled data
p = bokeh.plotting.figure(
    title="Audio Envelope (Downsampled)", 
    x_axis_label='Time (s)', 
    y_axis_label='Amplitude', 
    width=800, 
    height=400,
    x_range=(x_ds.min(), x_ds.max()),
    y_range=(min(y_ds.min(), audio_envelope_ds.min()), max(y_ds.max(), audio_envelope_ds.max()))
)
r = p.line(x_ds, y_ds, line_width=2, color='blue', legend_label='Sine Wave')
r2 = p.line(x_ds, audio_envelope_ds, line_width=2, color='red', legend_label='Audio Envelope')

bokeh.io.show(p)
# ...existing code...

## Create a rough transcription for alignment

In [ ]:

def window_rms(a, window_size):
  a2 = np.power(a,2)
  window = np.ones(window_size)/float(window_size)
  return np.sqrt(np.convolve(a2, window, 'valid'))   # same keeps border effects

# https://kferg.dev/posts/2020/audio-reactive-programming-envelope-followers/
def envelope_follower(signal, sample_rate, attack_time_ms, release_time_ms):
    # Convert attack/release times to filter coefficients
    alpha_attack = np.exp(-1 / (sample_rate * attack_time_ms / 1000))
    alpha_release = np.exp(-1 / (sample_rate * release_time_ms / 1000))

    envelope = np.zeros_like(signal, dtype=np.double)
    current_envelope = 0.0

    for i in range(len(signal)):
        abs_sample = abs(signal[i])
        if abs_sample > current_envelope:
            current_envelope = alpha_attack * current_envelope + (1 - alpha_attack) * abs_sample
        else:
            current_envelope = alpha_release * current_envelope + (1 - alpha_release) * abs_sample
        envelope[i] = current_envelope
    return envelope

def sensor_data_transformations(data, methods=[], outlier_method="zscore", 
                                  outlier_threshold=100, attack_time_ms=5, release_time_ms=500, v=False):
  transformed_signals = []
  mag = np.sqrt(np.array(data[1])**2 + np.array(data[2])**2 + np.array(data[3])**2)
  mask = detect_outliers(mag, method=outlier_method, threshold=outlier_threshold, v=v)
  mag_cleaned = mag[mask]
  timestamps_cleaned = np.array(data[0])[mask]
  x_cleaned = np.array(data[1], dtype=np.double)[mask]
  y_cleaned = np.array(data[2], dtype=np.double)[mask]
  z_cleaned = np.array(data[3], dtype=np.double)[mask]

  for method in methods:
    if method == "raw_x":
      transformed_signals.append((timestamps_cleaned, x_cleaned))
    elif method == "raw_y":
      transformed_signals.append((timestamps_cleaned, y_cleaned))
    elif method == "raw_z":
      transformed_signals.append((timestamps_cleaned, z_cleaned))
    elif method == "magnitude":
      transformed_signals.append((timestamps_cleaned, mag_cleaned))
    elif method == "rms":
      rms = window_rms(mag_cleaned, 64)
      rms -= np.median(rms)
      rms /= np.max(rms)
      timestamps_cleaned = timestamps_cleaned[:len(rms)]  # Adjust timestamps to match RMS length
      transformed_signals.append((timestamps_cleaned, rms))
    elif method == "envelope":
      # estimate sample rate
      sample_rate = 1 / (np.mean(np.diff(timestamps_cleaned)) / 1e6)
      # center and normalize z axis
      z_cleaned -= np.mean(z_cleaned)
      z_cleaned /= np.max(np.abs(z_cleaned))
      # Apply envelope follower
      # defaults: 5, 50
      transformed_signals.append((
        timestamps_cleaned,
        envelope_follower(z_cleaned, sample_rate, attack_time_ms=attack_time_ms, release_time_ms=release_time_ms)))
    elif method == "pca":
      pca = PCA(n_components=1)
      # Stack the x, y, z data for PCA
      data_matrix = np.vstack((x_cleaned, y_cleaned, z_cleaned)).T
      pca_result = pca.fit_transform(data_matrix)
      # Normalize the PCA result
      pca_result -= np.mean(pca_result)
      pca_result /= np.max(np.abs(pca_result))
      transformed_signals.append((timestamps_cleaned, pca_result.flatten()))
  return transformed_signals

def transform_data_and_find_peaks(data, 
                                  threshold=0.3, 
                                  methods=["rms"], 
                                  outlier_method="zscore", 
                                  outlier_threshold=100, 
                                  attack_time_ms=5, 
                                  release_time_ms=500,
                                  v=False):

    transformations = sensor_data_transformations(data, 
                                methods=methods, 
                                outlier_method="zscore", 
                                outlier_threshold=100, 
                                attack_time_ms=attack_time_ms, 
                                release_time_ms=release_time_ms, 
                                v=v)
    
    timestamps_cleaned, transformed_signal = transformations[0]
    
    peaks = find_peaks(transformed_signal, threshold)
    peaks = [timestamps_cleaned[i] for i in peaks]

    return transformed_signal, peaks, timestamps_cleaned, transformed_signal, transformations

def find_peaks(data, threshold=0.3):
    peaks = librosa.util.peak_pick(data, pre_max=100, post_max=100, pre_avg=1000, post_avg=1000, delta=0.1, wait=20)
    
    valid_peaks = []
    for i in peaks:
        if data[i] > threshold:
            valid_peaks.append(i)
    return valid_peaks


def process_sensor_data(
      filename, 
      methods=["rms"],
      threshold=0.3,
      attack_time_ms=5, 
      release_time_ms=500, 
      v=False, 
      plot=False):
  all_data, interrupt_timestamps = read_sensor_data_hex(filename, v=v)

  # get min of all timestamps in all_data
  global_min_time = min(min(sensor[0]) for sensor in all_data)

  if plot:
    p = bokeh.plotting.figure(
                title="Sensor Data and Energy Functions",
                x_axis_label="Time (s)",
                y_axis_label="Magnitude",
                width=1000,
                height=600,
            )
    colors = bokeh.palettes.Turbo256


  all_peaks = []
  all_sensor_data = []
  all_transformed_data = []
  all_timestamps = []
  for i, sensor_data in enumerate(all_data):

    transformations = sensor_data_transformations(sensor_data, 
                            methods=methods, 
                            outlier_method="zscore", 
                            outlier_threshold=100, 
                            attack_time_ms=attack_time_ms, 
                            release_time_ms=release_time_ms, 
                            v=v)
    
    timestamps_cleaned, transformed_signal = transformations[0]
    
    
    peaks = find_peaks(transformed_signal, threshold)
    peaks = [timestamps_cleaned[i] for i in peaks]

    all_peaks.append(peaks)
    all_transformed_data.append(transformed_signal)
    all_timestamps.append(timestamps_cleaned)
    all_sensor_data.append(transformed_signal)  # TODO: not the right thing

    if plot:
        # Plot sensor data
        for method, (timestamps, signal) in zip(methods, transformations):
          legend_label = f"Sensor {i + 1} - {method}"
          r = p.line(list(timestamps), signal, legend_label=legend_label, color=colors[i * 10 + 1])
          r.visible = False
  
  global_min_time = min(min(sensor) for sensor in all_timestamps)
  global_max_time = max(max(sensor) for sensor in all_timestamps)

  if plot: 
    p.legend.ncols = len(methods)
    p.legend.click_policy = "hide"
    p.add_layout(p.legend[0], 'right')
    bokeh.io.show(p)
    
  return interrupt_timestamps, all_sensor_data, all_transformed_data, all_peaks, all_timestamps, global_min_time, global_max_time



### midi transcription

In [ ]:
def generate_midi_from_peaks(all_peaks, output_file, ticks_per_beat=480, tempo=500000, note_length_ticks=100):
    """
    Generate a MIDI file from peak data with proper note lengths.

    Parameters:
    - all_peaks: list of lists, detected peaks for each bar of the balafon.
    - min_time: int, minimum timestamp across all sensors.
    - max_time: int, maximum timestamp across all sensors.
    - output_file: str, path to save the generated MIDI file.
    - ticks_per_beat: int, resolution of the MIDI file in ticks per beat.
    - tempo: int, tempo in microseconds per beat (default is 500,000 for 120 BPM).
    - note_length_ticks: int, duration of each note in ticks.
    """
    # Flatten and sort all peaks by time
    events = []
    seconds_per_beat = tempo / 1e6
    for i, peak_list in enumerate(all_peaks):
        for peak in peak_list:
            # Convert peak time (in seconds) to ticks
            current_tick = int((peak / seconds_per_beat) * ticks_per_beat)
            # Schedule note_on and note_off events
            events.append((current_tick, 'note_on', MIDI_NOTE_NUMS[i], 64))  # Note on
            events.append((current_tick + note_length_ticks, 'note_off', MIDI_NOTE_NUMS[i], 64))  # Note off

    # Sort all events by absolute tick time
    events.sort(key=lambda x: x[0])

    # Create the MIDI file and track
    mid = mido.MidiFile(ticks_per_beat=ticks_per_beat)
    track = mido.MidiTrack()
    mid.tracks.append(track)

    # Add tempo meta message
    track.append(mido.MetaMessage('set_tempo', tempo=tempo))

    # Add events to the MIDI track with correct relative timing
    last_tick = 0
    for tick, event_type, note, velocity in events:
        delta_ticks = tick - last_tick
        last_tick = tick
        track.append(mido.Message(event_type, note=note, velocity=velocity, time=delta_ticks))

    # Save the MIDI file
    mid.save(output_file)


def generate_midi_file_from_initial_data(data_file, output_file, ticks_per_beat=480, tempo=500000, note_length_ticks=100):
    """
    Generate a MIDI file from sensor data with proper note lengths.

    Parameters:
    - data_file: str, path to the input data file.
    - output_file: str, path to save the generated MIDI file.
    - ticks_per_beat: int, resolution of the MIDI file in ticks per beat.
    - tempo: int, tempo in microseconds per beat (default is 500,000 for 120 BPM).
    - note_length_ticks: int, duration of each note in ticks.
    """
    _, _, _, all_peaks, _, min_time, max_time = process_sensor_data(data_file, threshold=0.1)
    generate_midi_from_peaks(all_peaks, min_time, max_time, output_file, ticks_per_beat, tempo, note_length_ticks)

## Bandpass filters

In [ ]:
# def cqt(audio, center_freqs, sr=None, hop_length=32, bins_per_octave=20):
#     """
#     Compute the Constant-Q Transform (CQT) of an audio signal and return the CQT matrix.

#     Parameters:
#     - audio: audio signal as a numpy array
#     - center_freqs: list of floats, specific center frequencies for the filterbank.
#     - sr: sampling rate of the audio
#     - hop_length: int, number of samples between successive frames for the CQT.
#     - bins_per_octave: int, number of bins per octave in the CQT.

#     Returns:
#     - timestamps: numpy array, timestamps corresponding to the CQT channels.
#     - cqt_channels: list of numpy arrays, CQT channels corresponding to the specified center frequencies.
#     """
#     n_bins = bins_per_octave * int(np.ceil(np.log2(max(center_freqs) / min(center_freqs))))
#     cqt = librosa.cqt(
#         audio, sr=sr, 
#         hop_length=hop_length, 
#         fmin=min(center_freqs), 
#         n_bins=n_bins, 
#         bins_per_octave=bins_per_octave)
#     cqt_frequencies = librosa.cqt_frequencies(cqt.shape[0], fmin=min(center_freqs), bins_per_octave=bins_per_octave)
    
#     cqt_channels = []
#     for freq in center_freqs:
#         if freq not in cqt_frequencies:
#             closest_bin = np.argmin(np.abs(cqt_frequencies - freq))
#             if np.abs(cqt_frequencies[closest_bin] - freq) > 1e-6:
#                 raise ValueError(f"Frequency {freq} not found in CQT frequencies. Closest is {cqt_frequencies[closest_bin]}.")
#             cqt_channels.append(cqt[closest_bin])
#     timestamps = np.arange(len(cqt_channels[0])) * hop_length / sr
#     return timestamps, cqt_channels

def compute_energy_with_full_cqt(audio, center_freqs, sr=None, hop_length=512, bins_per_octave=20):
    """
    Compute energy functions using the full Constant-Q Transform (CQT) and map to specific frequencies.

    Parameters:
    - audio: audio signal as a numpy array
    - center_freqs: list of floats, specific center frequencies for the filterbank.
    - sr: sampling rate of the audio
    - hop_length: int, number of samples between successive frames for the CQT.
    - bins_per_octave: int, number of bins per octave in the CQT.

    Returns:
    - timestamps: numpy array, timestamps corresponding to the energy functions.
    - energies: list of numpy arrays, energy functions for each center frequency.
    """

    # Compute the full CQT
    cqt = librosa.cqt(audio, sr=sr, hop_length=hop_length, fmin=min(center_freqs), n_bins=bins_per_octave * int(np.ceil(np.log2(max(center_freqs) / min(center_freqs)))), bins_per_octave=bins_per_octave)

    # Get the frequencies corresponding to the CQT bins
    cqt_frequencies = librosa.cqt_frequencies(cqt.shape[0], fmin=min(center_freqs), bins_per_octave=bins_per_octave)

    # Map the desired frequencies to the closest CQT bins
    energies = []
    for freq in center_freqs:
        # Find the closest CQT bin
        closest_bin = np.argmin(np.abs(cqt_frequencies - freq))
        energy = np.abs(cqt[closest_bin])**2  # Energy is the squared magnitude
        energy -= np.median(energy)
        energy /= np.max(energy)
        energies.append(energy)
    
    # Compute timestamps for the energy function
    timestamps = np.arange(len(energies[0])) * hop_length / sr  # Energy timestamps in seconds

    return timestamps, energies

def plot_energy_functions_matplotlib(energies, sr, center_freqs):
    """
    Plot energy functions for each center frequency using Matplotlib.
    
    Parameters:
    - energies: list of numpy arrays, energy functions for each center frequency.
    - sr: int, sampling rate of the audio.
    - center_freqs: list of floats, center frequencies for the filterbank.
    """
    time = np.linspace(0, len(energies[0]) / sr, len(energies[0]))
    plt.figure(figsize=(12, 8))
    for i, energy in enumerate(energies):
        plt.plot(time, energy, label=f"{center_freqs[i]} Hz")
    plt.xlabel("Time (s)")
    plt.ylabel("Energy")
    plt.title("Energy Functions for Each Center Frequency")
    plt.legend()
    plt.show()

def process_audio_with_filterbank(audio_file, center_freqs, filter_size=16, plot=False):
    """
    Process an audio file with a filterbank and plot energy functions.
    
    Parameters:
    - audio_file: str, path to the audio file.
    - center_freqs: list of floats, center frequencies for the filterbank.
    - filter_size: int, size of the moving average filter to smooth the energy functions.
    - plot: bool, whether to plot the energy functions.
    """
    # Load the audio file
    audio, sr = librosa.load(audio_file)
    
    # Filter the audio and compute energy functions
    energy_timestamps, energies = compute_energy_with_full_cqt(audio, center_freqs, sr)

    for i, energy in enumerate(energies):
        # moving average of the energy function
        energy = np.convolve(energy, np.ones(filter_size)/filter_size, mode='same')
        energies[i] = energy

    
    # Plot the energy functions
    if plot:
        plot_energy_functions_matplotlib(energies, sr, center_freqs)
    
    return energy_timestamps, energies


In [ ]:
energy_timestamps, energies = process_audio_with_filterbank("audio/ZOOM0004_INPUT1_TRIMMED.WAV", BALAFON_TUNING, plot=True)

## Dynamic thresholding

In [ ]:
def find_peaks_dynamic(data):
    """
    Find peaks in the data using dynamic thresholds based on energy.

    Parameters:
    - data: numpy array, the input data (e.g., RMS values).

    Returns:
    - valid_peaks: list, indices of valid peaks.
    """
    # Use librosa's peak_pick for initial peak detection
    peaks = librosa.util.peak_pick(data, pre_max=50, post_max=50, pre_avg=50, post_avg=50, delta=0.01, wait=5)

    # # Validate peaks using the dynamic threshold
    # valid_peaks = []
    # for i in peaks:
    #     if data[i] > 1 - (alpha * energy[i]):  # Compare to scaled energy threshold
    #         valid_peaks.append(i)
    return peaks

def process_sensor_data_with_thresholds(sensor_data, timestamps_shifted, energies, energy_timestamps, alpha, plot=False):
    if plot:
        # use bokeh to plot sensor data, energy functions, and detected peaks in a tabbed layout
        p = bokeh.plotting.figure(
            title="Sensor Data and Energy Functions",
            x_axis_label="Time (s)",
            y_axis_label="Magnitude",
            width=1000,
            height=600,
        )
        colors = bokeh.palettes.Turbo256

    processed_data = []
    all_peaks = []

    for i, (sensor, sensor_timestamps, band_energy) in enumerate(zip(sensor_data, timestamps_shifted, energies)):
        # # clip energy function
        # band_energy = np.minimum(0.75, (band_energy * alpha))
        # band_energy = np.maximum(0.1, band_energy) 
        
        # # Interpolate energy function to match sensor timestamps
        # energy_interpolated = np.interp(sensor_timestamps, energy_timestamps, band_energy)

        # diff = np.maximum(0, sensor - energy_interpolated)
        # processed_data.append(diff)
        sensor = np.maximum(0.4, sensor)

        peaks = find_peaks_dynamic(sensor)
        peak_timestamps = [sensor_timestamps[p] for p in peaks]
        all_peaks.append(peak_timestamps)

        if plot:
            # Plot sensor data
            r = p.line(list(sensor_timestamps), sensor, legend_label=str(i + 1), color=colors[i * 10 + 1])
            r.visible = False

            # Plot energy function
            # p.line(sensor_timestamps, energy_interpolated, legend_label=f"Energy {i+1}", color=colors[i * 10 + 1])
            # r.visible = False

            # Plot difference
            # r = p.line(sensor_timestamps, diff, legend_label=str(i + 1), color=colors[i * 10 + 1])

            # # Plot detected peaks
            r = p.vspan(x=peak_timestamps, legend_label=f"Peaks {i+1}", color="red")
            r.visible = False

    if plot: 
        p.legend.click_policy = "hide"
        p.legend.ncols = 2
        p.add_layout(p.legend[0], 'right')
        bokeh.io.show(p)
    
    return processed_data, all_peaks

def align_sensor_timestamps(all_timestamps, global_min_time):
    """
    Align sensor timestamps to start at the global minimum timestamp.

    Parameters:
    - all_timestamps: list of lists, each sublist contains timestamps for a sensor.
    - global_min_time: int, global minimum timestamp across all sensors.

    Returns:
    - aligned_timestamps: list of lists, aligned sensor timestamps.
    """
    aligned_timestamps = []
    for sensor_timestamps in all_timestamps:
        timestamps_shifted = sensor_timestamps - global_min_time
        timestamps_shifted = np.array(timestamps_shifted) / 1e6  # Convert to seconds
        aligned_timestamps.append(timestamps_shifted)
    return aligned_timestamps

In [ ]:
_, all_data, all_transformed_data, all_peaks, all_timestamps, global_min_time, global_max_time =  process_sensor_data(
    "data/test2", 
    methods=["envelope", "raw_z"], 
    attack_time_ms=5, 
    release_time_ms=800, 
    threshold=0.3,
    plot=True
)

energy_timestamps, energies = process_audio_with_filterbank("audio/ZOOM0004_INPUT1_TRIMMED.WAV", BALAFON_TUNING)

aligned_timestamps = align_sensor_timestamps(all_timestamps, global_min_time)

processed_sensor_data, all_peaks = process_sensor_data_with_thresholds(all_transformed_data, aligned_timestamps, energies, energy_timestamps, 10, plot=True)

generate_midi_from_peaks(all_peaks, 0, global_max_time, "out/test2_out2.mid")